In [ ]:
#| default_exp utils

# Utilities
> Various utilities

In [ ]:
#| export
from pathlib import Path
from netCDF4 import Dataset
from fastcore.all import *
import pandas as pd
import numpy as np
from ast import literal_eval

from marisco.configs import NC_VARS

We define below useful constants throughout the package.

## NetCDF Utilities

Extract NetCDF contents

In [ ]:
#| export
# way too long for our coding standard - REFACTORING NEEDED
class ExtractNetcdfContents:
    def __init__(self, filename: str, verbose: bool = False):
        "Initialize and extract data from a NetCDF file."
        self.filename = filename
        self.verbose = verbose
        self.dfs = {}  # DataFrames extracted from the NetCDF file
        self.enum_dicts = {}  # Enum dictionaries extracted from the NetCDF file
        self.global_attrs = {}  # Global attributes extracted from the NetCDF file
        self.custom_maps = {}  # Custom maps extracted from the NetCDF file
        self.extract_all()

    def extract_all(self):
        "Extract data, enums, and global attributes from the NetCDF file."
        if not Path(self.filename).exists():
            print(f'File {self.filename} not found.')
            return
        
        with Dataset(self.filename, 'r') as nc:
            self.global_attrs = self.extract_global_attributes(nc)
            for group_name in nc.groups:
                group = nc.groups[group_name]
                self.dfs[group_name.upper()] = self.extract_data(group)
                self.enum_dicts[group_name.upper()] = self.extract_enums(group, group_name)
                self.custom_maps[group_name.upper()] = self.extract_custom_maps(group, group_name)
                
            if self.verbose:
                print("Data extraction complete.")

    def extract_data(self, group) -> pd.DataFrame:
        "Extract data from a group and convert to DataFrame."
        data = {var_name: var[:] for var_name, var in group.variables.items() if var_name not in group.dimensions}
        df = pd.DataFrame(data)
        rename_map = {nc_var: col for col, nc_var in NC_VARS.items() if nc_var in df.columns}
        df = df.rename(columns=rename_map)
        return df

    def extract_enums(self, group, group_name: str) -> Dict:
        "Extract enum dictionaries for variables in a group."
        local_enum_dicts = {}
        for var_name, var in group.variables.items():
            if hasattr(var.datatype, 'enum_dict'):
                local_enum_dicts[var_name] = {str(k): str(v) for k, v in var.datatype.enum_dict.items()}
                if self.verbose:
                    print(f"Extracted enum_dict for {var_name} in {group_name}")
        return local_enum_dicts

    def extract_global_attributes(self, nc) -> Dict:
        "Extract global attributes from the NetCDF file."
        globattrs = {attr: getattr(nc, attr) for attr in nc.ncattrs()}
        return globattrs
    
    def extract_custom_maps(self, group, group_name: str) -> Dict:
        "Extract custom maps from the NetCDF file."
        reverse_nc_vars = {v: k for k, v in NC_VARS.items()}        
        custom_maps = {}
        for var_name, var in group.variables.items():
            attr=f"{var_name}_map"
            if hasattr(var, attr):
                custom_maps[reverse_nc_vars[var.name]] =  literal_eval(getattr(var, attr))
        return custom_maps

In [ ]:
#| eval: false
# fname = Path('../../_data/output/190-geotraces-2021.nc')
#fname = Path('../../_data/output/tepco.nc')
#fname = Path('../../_data/output/tepco.nc')
#fname = Path('./files/nc/encoding-test.nc')
#contents= ExtractNetcdfContents(fname)
#print(contents.dfs)
#print(contents.enum_dicts)
#print(contents.global_attrs)
#print(contents.custom_maps)

In [ ]:
!ls 

_netcdf2csv.ipynb  data		   files	metadata.ipynb
callbacks.ipynb    decoders.ipynb  geo.ipynb	nc2csv.ipynb
configs.ipynb	   encoders.ipynb  match.ipynb	utils.ipynb


A junior dev developped this module which is essentially reading a netcdf with multiple groups (the one generated out of the "encoders" module via nbs/handlers curation pipeline for individual data providers data.

it's mainly used in the "nc2csv.ipynb" module. 

This class is now in a "utils" module consisting of this single class "ExtractNetcdfContents". Several points/questions:

- this class is converting back netcdf file containing multiple groups + enums + global attributes and provide a set of class attributes allowing to access them (each group data as df in a dfs dict - mimicking the data structure used to encode into netcdf files). what could be a better name for such class/function, what would be a better name for it, a better location (does it deserve its own module), ...

Always answer from the perspective of J.Howard/fastai/fastcore mindset.

J. Howard/fastcore is firmly against using dataclass, why and what's the proposed alterative?

- yes makes a lot of sense. Also would be great to have a nice __repr__ method to display succintly and also in an informed way netcdf content. 
- before we think about a blueprint of the rewriting, check the "nc2csv.ipynb" module as it's the main client. Also consult the optional CRAFTs file on sicp design guidelines

yes if we also develop helpers to read and visualize global attrs, retrieve and print nicely specific enums, but also group's variable attributes, we could put in a dedicate moodule (but with a sound name)

not sure! what we do is essentially reading a produced nc (i) in a format useful for other modules like nc2csv but also (ii) to be able to read/print its content for inspection, ...

i like the // with the unix-like ncdum that I use but it could be also confusing because it will be only understood by people already knowing this tool. Exactly the king of audience that could actually already use the original ncdump unix command -:) Maybe something along the line of "nc_reader" much better

yes let's discuss the public interface

often J. Howard/fastai/fastcore/answerai don't have those dedicated show() helper right, just return class with `__repr__` naturally displayed when returned. do you see what i mean?

yes but in that latter case, we would read everything at once right even if we were only interested in global attrs

yes maybe this second case

yes ok something along that line. but now we have a rough plan let's roll up our sleeves and implement it line but line following the solveit, ideally the right abstraction will emerge.

In [ ]:
!ls files/nc

100_HELCOM_MORS_2024.nc		   encoding-test.nc
100_HELCOM_MORS_2024_BIOTA.csv	   maris-cdl.nc
100_HELCOM_MORS_2024_SEAWATER.csv  maris-template.nc
100_HELCOM_MORS_2024_SEDIMENT.csv  template-test.nc


ok we can use "00_HELCOM_MORS_2024.nc" file for testing purpose

In [ ]:
f = Dataset('files/nc/100_HELCOM_MORS_2024.nc', 'r')

In [ ]:
f.groups.keys()

dict_keys(['biota', 'seawater', 'sediment'])

In [ ]:
f.ncattrs()

['id',
 'title',
 'summary',
 'keywords',
 'history',
 'keywords_vocabulary',
 'keywords_vocabulary_url',
 'record',
 'featureType',
 'cdm_data_type',
 'Conventions',
 'publisher_name',
 'publisher_email',
 'publisher_url',
 'publisher_institution',
 'creator_name',
 'institution',
 'metadata_link',
 'creator_email',
 'creator_url',
 'references',
 'license',
 'comment',
 'geospatial_lat_min',
 'geospatial_lon_min',
 'geospatial_lat_max',
 'geospatial_lon_max',
 'geospatial_vertical_min',
 'geospatial_vertical_max',
 'geospatial_bounds',
 'geospatial_bounds_crs',
 'time_coverage_start',
 'time_coverage_end',
 'local_time_zone',
 'date_created',
 'date_modified',
 'publisher_postprocess_logs']

In [ ]:
grp = f.groups['seawater']
print("seawater vars:", list(grp.variables.keys()))
print("Dim names:", list(grp.dimensions.keys()))
# Show a few variable names + their attributes

seawater vars: ['id', 'id_provider', 'lon', 'lat', 'smp_depth', 'tot_depth', 'time', 'station', 'nuclide', 'value', 'unit', 'unc', 'dl', 'filt', 'sal', 'temp']


Dim names: ['id']


In [ ]:
for vn in list(grp.variables.keys())[:5]:
    v = grp.variables[vn]
    print(f"  {vn}: dtype={v.datatype}, dims={v.dimensions}")
    # check for custom_map
    print(f"    attrs: {list(v.ncattrs())}")

f.close()

  id: dtype=uint64, dims=('id',)


    attrs: ['long_name']


  id_provider: dtype="<class 'netCDF4.VLType'>": string type, dims=('id',)


    attrs: ['long_name']


  lon: dtype=float32, dims=('id',)


    attrs: ['long_name', 'standard_name', 'units']


  lat: dtype=float32, dims=('id',)


    attrs: ['long_name', 'standard_name', 'units']


  smp_depth: dtype=float32, dims=('id',)


    attrs: ['long_name', 'standard_name', 'units', 'axis']


ok looking staightforward

yes

In [ ]:
def read_nc_attrs(fname):
    "Return global attributes of a MARIS NetCDF file as a dict."
    with Dataset(fname, 'r') as nc:
        return {a: getattr(nc, a) for a in nc.ncattrs()}

then?

In [ ]:
read_nc_attrs('files/nc/100_HELCOM_MORS_2024.nc')

{'id': '26VMZZ2Q',
 'title': 'Environmental database - Helsinki Commission Monitoring of Radioactive Substances',
 'summary': 'MORS Environment database has been used to collate data resulting from monitoring of environmental radioactivity in the Baltic Sea based on HELCOM Recommendation 26/3.\n\nThe database is structured according to HELCOM Guidelines on Monitoring of Radioactive Substances (https://www.helcom.fi/wp-content/uploads/2019/08/Guidelines-for-Monitoring-of-Radioactive-Substances.pdf), which specifies reporting format, database structure, data types and obligatory parameters used for reporting data under Recommendation 26/3.\n\nThe database is updated and quality assured annually by HELCOM MORS EG.',
 'keywords': 'oceanography, Earth Science > Oceans > Ocean Chemistry> Radionuclides, Earth Science > Human Dimensions > Environmental Impacts > Nuclear Radiation Exposure, Earth Science > Oceans > Ocean Chemistry > Ocean Tracers, Earth Science > Oceans > Marine Sediments, Eart

how could we print it nicely for instance?

let's try those two options?

In [ ]:
import pprint
attrs = read_nc_attrs('files/nc/100_HELCOM_MORS_2024.nc')
pprint.pprint(attrs)

{'Conventions': 'CF-1.10 ACDD-1.3',
 'cdm_data_type': 'TBD',
 'comment': 'TBD',
 'creator_email': 'TBD',
 'creator_name': '[{"creatorType": "author", "name": "HELCOM MORS"}]',
 'creator_url': 'TBD',
 'date_created': 'TBD',
 'date_modified': 'TBD',
 'featureType': 'TBD',
 'geospatial_bounds': 'POLYGON ((10.2917 54.006167, 29.05 54.006167, 29.05 '
                      '60.3767, 10.2917 60.3767, 10.2917 54.006167))',
 'geospatial_bounds_crs': 'EPSG:4326',
 'geospatial_lat_max': '60.3767',
 'geospatial_lat_min': '54.006167',
 'geospatial_lon_max': '29.05',
 'geospatial_lon_min': '10.2917',
 'geospatial_vertical_max': '72.0',
 'geospatial_vertical_min': '0.0',
 'history': 'TBD',
 'id': '26VMZZ2Q',
 'institution': 'TBD',
 'keywords': 'oceanography, Earth Science > Oceans > Ocean Chemistry> '
             'Radionuclides, Earth Science > Human Dimensions > Environmental '
             'Impacts > Nuclear Radiation Exposure, Earth Science > Oceans > '
             'Ocean Chemistry > Ocean Trace

In [ ]:
import json
print(json.dumps(attrs, indent=2))

{
  "id": "26VMZZ2Q",
  "title": "Environmental database - Helsinki Commission Monitoring of Radioactive Substances",
  "summary": "MORS Environment database has been used to collate data resulting from monitoring of environmental radioactivity in the Baltic Sea based on HELCOM Recommendation 26/3.\n\nThe database is structured according to HELCOM Guidelines on Monitoring of Radioactive Substances (https://www.helcom.fi/wp-content/uploads/2019/08/Guidelines-for-Monitoring-of-Radioactive-Substances.pdf), which specifies reporting format, database structure, data types and obligatory parameters used for reporting data under Recommendation 26/3.\n\nThe database is updated and quality assured annually by HELCOM MORS EG.",
  "keywords": "oceanography, Earth Science > Oceans > Ocean Chemistry> Radionuclides, Earth Science > Human Dimensions > Environmental Impacts > Nuclear Radiation Exposure, Earth Science > Oceans > Ocean Chemistry > Ocean Tracers, Earth Science > Oceans > Marine Sediments

how could we improve it more? markdown?

show me a markdown example?

**100_HELCOM_MORS_2024.nc**

Groups: seawater (2842 rows), sediment (423 rows), biota (1847 rows)

**Global attrs** (key ones):
| Attr | Value |
|---|---|
| title | Environmental database - Helsinki Commission ... |
| publisher_name | Paul MCGINNITY, Iolanda OSVATH, ... |
| time_coverage_start | 1985-07-16 |
| time_coverage_end | 2023-06-11 |
| geospatial_bounds | POLYGON ((10.2917 54.006167, 29.05 54.006167, ... |
| date_created | TBD |
| Conventions | CF-1.10 ACDD-1.3 |
| ... (28 more) |

Actually, I think i'll postpone the refactoring of this module and just write an helper function that given a netcdf file path, returns the corresponding dfs with key/group as expected by nc2csv module. I think it's mainly a function implementing more or less "extract_data" in this module.

- check what's expected in nbs/api/nc2csv.ipynb
- let's writ a simple helper function returning this dfs
- maybe let's refactor slightly the nc2csv module, now accessing dfs via an attribute of `contents.dfs` but we could simply return dfs

In [ ]:
def read_nc_groups(fname):
    "Read a MARIS NetCDF file and return {group: DataFrame} dict."
    with Dataset(fname, 'r') as nc:
        dfs = {}
        for gn, grp in nc.groups.items():
            data = {vn: grp.variables[vn][:] for vn in grp.variables
                    if vn not in grp.dimensions}
            df = pd.DataFrame(data)
            rename = {n: k for k, n in NC_VARS.items() if n in df.columns}
            dfs[gn.upper()] = df.rename(columns=rename)
    return dfs

In [ ]:
# instead of:
contents = ExtractNetcdfContents(fname_in)
dfs = keep_csv_cols(contents.dfs)

# just:
dfs = keep_csv_cols(read_nc_groups(fname_in))